# Core 09 - Multi Agentic System

Objetivo: coordinar dos Agents dentro del mismo System y componer exclusivamente sus `RunResult` reales.

**Lugar en el modelo:** el System conecta dos Agents y su `SequentialPlan` decide qué recibe la segunda unidad.

**Evidencia exigida:** `system.run()` debe producir un `RunResult` compuesto con dos hijos reales y transferencia observable de evidencia.

**Límite de la evidencia:** la secuencia es un plan explícito; no se atribuye coordinación mágica al Provider o al Framework.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_MULTI_PROVIDER | python-runtime | Compartir provider entre agentes. |
| symbols | tool y graph | Distribuir entradas reales por especialista. |
| aggregation | compose_result | Unir RunResult sin fabricar evidencia. |

## 1) Runtime compartido

El provider es seleccionable mediante `AGENTIC_SYSTEMS_MULTI_PROVIDER`; el default local permite ejecutar todo el notebook sin credenciales.

In [ ]:
import os

import agentic_systems as toolkit

PROVIDER = os.getenv("AGENTIC_SYSTEMS_MULTI_PROVIDER", "python-runtime")
runtime = toolkit.runtime(provider=PROVIDER)
system = toolkit.system(runtime=runtime)
toolkit.show_json(runtime.describe(), title="Multi-system runtime")

## 2) Tools y Agents con responsabilidades separadas

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

@toolkit.tool
def review_observation(symbol: str, is_public: bool) -> dict:
    return {
        "symbol": symbol,
        "accepted": isinstance(is_public, bool),
        "observed_public_value": is_public,
    }

inspector = system.agent(
    name="multi_system_inspector",
    instructions="Inspecciona el simbolo con la Tool requerida.",
    tools=[inspect_public_api], runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
reviewer = system.agent(
    name="multi_system_reviewer",
    instructions="Revisa la observacion con la Tool requerida.",
    tools=[review_observation], runtime=runtime,
    contract=toolkit.AgentContract(must_call=["review_observation"]),
)

## 3) Compilar y ejecutar la secuencia

`SequentialPlan.input_selector` convierte el `RunResult` del Inspector en el input del Reviewer. `system.run()` ejecuta ambos Agents; el notebook no coordina el flujo con dos llamadas manuales.

In [ ]:
symbol = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "agent")
inspect_request = (
    {"tool": "inspect_public_api", "input": {"symbol": symbol}}
    if PROVIDER == "python-runtime"
    else f"Usa inspect_public_api para verificar {symbol}."
)

def review_input(child_result):
    fields = toolkit.agent_output(child_result)["fields"]
    if "is_public" not in fields:
        return fields
    if PROVIDER == "python-runtime":
        return {
            "tool": "review_observation",
            "input": {"symbol": fields["symbol"], "is_public": fields["is_public"]},
        }
    return (
        f"Usa review_observation con symbol={fields['symbol']} "
        f"e is_public={fields['is_public']}."
    )

execution = toolkit.SequentialPlan(input_selector=review_input)
compiled = system.compile(execution=execution, name="multi_agent_system")
assert compiled.inspect() == {
    "name": "multi_agent_system",
    "execution_plan": "sequential",
    "unit_count": 2,
}
result = system.run(inspect_request, execution=execution, mode="eval")
assert result.ok, result.errors
assert len(result.children) == 2

inspection_run, review_run = result.children
inspection_output = toolkit.agent_output(inspection_run)
review_output = toolkit.agent_output(review_run)
assert review_output["fields"]["symbol"] == inspection_output["fields"]["symbol"]

toolkit.human_result(result, title="Multi-agent System RunResult", show_lineage=True)
toolkit.show_json(
    {"inspection": inspection_output, "review": review_output},
    title="Agent outputs",
)

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "toolkit.tool", "system.agent",
    "toolkit.SequentialPlan", "system.compile", "CompiledSystem.inspect", "system.run",
    "toolkit.agent_output",
    "toolkit.human_result", "toolkit.show_json",
    "toolkit.AgentContract",
]
toolkit.show_json(api_coverage, title="Multi-system API coverage")

## Resultado e interpretacion

Un `RunResult` del System con dos hijos reales. Runtime, usage, validation y tool events proceden del Inspector y Reviewer ejecutados por el plan secuencial.